# 02 Clean the data, then split it

**What this notebook does.** Takes out the wafers with a 3 and the extra copies from
notebook 01. Then cuts clean wafers into three parts. Training (70%) is what the model learns from.
Validation (15%) is a practice test to try settings on.

**Why the order matters.** Cleaning has to come before splitting. If I split first a wafer could end up in
training and its copy in test. Removing one of them afterwards doesn't undo that the model already learned from it.

**In:** `data/Wafer_Map_Datasets.npz` **Out:** `output/split.npz`\

## 1. Load, and make it lighter

**Why this step:** The duplicate check makes a second copy of all maps in memory. Making the arrays smaller first keeps my laptop from running out of memory.

**The problem:** Load the maps and labels, using as little memory as possible.

**How:** The maps come in as `int32`: 4 bytes per square, a few hundred MB. The values only go up to 3, and `int8` holds up to 127 in 1 byte. `.astype(np.int8)` switches type. Nothing changes in the values, it's just four times smaller. `.nbytes` shows how much memory an array takes.

In [1]:
import os

import numpy as np
import wafer

data = np.load("../data/Wafer_Map_Datasets.npz")

maps = data["arr_0"].astype(np.int8)      # 0 to 3 fits easily in int8
labels = data["arr_1"].astype(np.int8)

print(f"Maps original size: {data["arr_0"].nbytes / 1e6:.0f} MB")
print("Maps:", maps.shape, maps.dtype, f"{maps.nbytes / 1e6:.0f} MB")
print("Labels:", labels.shape, labels.dtype)

Maps original size: 411 MB
Maps: (38015, 52, 52) int8 103 MB
Labels: (38015, 8) int8


**What you should see:** The maps as `int8`, four times smaller in memory than before.

## 2. Take out the wafers with a 3 (FR-04)

**Why this step:** A 3 is something I can't explain by reading the documentation, so it shouldn't end up in the numbers the model learns from.

**The problem:** Remove every wafer that has at least one 3.

**How:** `has_3` is True per wafer with a 3. `assert something, "message"` stops the notebook with that message if the something is False. I use it as a check that can't be missed, unlike a print.

In [2]:
has_3 = (maps == 3).any(axis=(1, 2))
keep = ~has_3 # True for every wafer without a 3
print(f"Wafers before: {len(maps):,}")
print(f"Wafers with a 3: {has_3.sum():,}")

maps = maps[keep]
labels = labels[keep]

print(f"Wafers left: {len(maps):,}")
print("Values left in the maps:", np.unique(maps))

# Check: only 0, 1 and 2 may be left
assert set(np.unique(maps)) <= {0, 1, 2}, "a value other than 0, 1 or 2 is still in there"

Wafers before: 38,015
Wafers with a 3: 105
Wafers left: 37,910


Values left in the maps: [0 1 2]


## 3. Take out the extra copies (FR-03)

**Why this step:** This is the data leakage fix.

**The problem:** Keep one of every map and drop the extra copies.

**How:** `np.unique(..., return_index=True)` gives the position where it shows up first per map.\
`np.unique` sorts its output by the maps themselves, so those positions come back mixed up. `np.sort` puts them back in file order, which keeps the wafers in the same order as before.\
**First a check:** keeping the first copy only works if every copy of a map carries the same label. `np.concatenate(..., axis=1)` glues each label to the end of its map, and `np.unique` counts the different rows again. If there are more map-and-label rows than maps, some map has two labels, and keeping the first one would be a guess. Then the notebook stops, and that's a decision to make before going on.

In [3]:
# One row per map, so whole maps can be compared
maps_flat = maps.reshape(len(maps), -1)
unique_maps, first_index = np.unique(maps_flat, axis=0, return_index=True)

# Check if all copies of one map carry the same label, so keeping the first copy keeps the right label
maps_and_labels = np.concatenate([maps_flat, labels], axis=1)
assert len(np.unique(maps_and_labels, axis=0)) == len(unique_maps), "copies of one map have different labels"

# Back into file order
first_index = np.sort(first_index)

print(f"Wafers before: {len(maps):,}")
print(f"Different maps: {len(first_index):,}")
print(f"Copies removed: {len(maps) - len(first_index):,}")

maps = maps[first_index]
labels = labels[first_index]
print(f"Wafers left: {len(maps):,}")

# Check if every map is in there once
check_flat = maps.reshape(len(maps), -1)
assert len(np.unique(check_flat, axis=0)) == len(maps), "a map is still in there twice"

Wafers before: 37,910
Different maps: 36,890
Copies removed: 1,020
Wafers left: 36,890


## 4. Split 70 / 15 / 15 per type with seed 42 (FR-05)

**Why this step:** This decides which wafers are used where.

**The problem:** Cut the wafers into three parts, and make sure every type is in every part.

**How:** First per type, the smallest type Near-Full has 149 wafers by the description, and a few less after cleaning. A random split over all 36,890 could leave few or none in the test part, and then I can't score that pattern. So I did it per type, take all wafers of one type and shuffle them, give 70% to training, 15% to validation and 15% to test. Then the next type.\
**Seed 42:** shuffling uses random numbers. `np.random.default_rng(42)` makes them the same every run, so the split is the same every run (QC-06). If a score changes later it's because of the model, not because of random chance in the split.\
`rng.shuffle(rows)` mixes up a list in place. `list.extend` adds a whole list to the end of another one.

In [4]:
SEED = 42
rng = np.random.default_rng(SEED)
train_idx = []
val_idx = []
test_idx = []

# Every different label combination is one type
combos = np.unique(labels, axis=0)

for combo in combos:
    # The positions of every wafer of this type
    rows = np.where((labels == combo).all(axis=1))[0]
    rng.shuffle(rows)

    n_train = int(len(rows) * 0.70)
    n_val = int(len(rows) * 0.15) # test gets the rest, so no wafer gets lost to rounding

    train_idx.extend(rows[:n_train])
    val_idx.extend(rows[n_train:n_train + n_val])
    test_idx.extend(rows[n_train + n_val:])

train_idx = np.sort(np.array(train_idx))
val_idx = np.sort(np.array(val_idx))
test_idx = np.sort(np.array(test_idx))

total = len(train_idx) + len(val_idx) + len(test_idx)
print(f"Training: {len(train_idx):,} ({len(train_idx) / total * 100:.1f}%)")
print(f"Validation: {len(val_idx):,} ({len(val_idx) / total * 100:.1f}%)")
print(f"Test: {len(test_idx):,} ({len(test_idx) / total * 100:.1f}%)")

Training: 25,822 (70.0%)
Validation: 5,533 (15.0%)
Test: 5,535 (15.0%)


## 5. Prove the split

**Why this step:** If any of these checks fails every step after this is worthless. Asserts stop the notebook instead of carrying on with a broken split.

**The problem:** Check that no wafer and no map is in two parts, and that every type is in every part.

**How:** `np.intersect1d(a, b)` gives what two lists have in common. Empty means no wafer is in both.\
For maps: two different wafers could still have the same map. So I put the training maps and the test maps in one pile with `np.concatenate` and count the different ones. If that's lower than the pile, a map is in both.\
For types: per type I count the wafers in each part, and none may be 0.

In [5]:
# a) No wafer in two parts
assert len(np.intersect1d(train_idx, val_idx)) == 0, "a wafer is in training and validation"
assert len(np.intersect1d(train_idx, test_idx)) == 0, "a wafer is in training and test"
assert len(np.intersect1d(val_idx, test_idx)) == 0, "a wafer is in validation and test"
print("No wafer is in two parts")

# b) No map in training and test
flat = maps.reshape(len(maps), -1)
pile = np.concatenate([flat[train_idx], flat[test_idx]])
assert len(np.unique(pile, axis=0)) == len(pile), "the same map is in training and test"
print("No map is in training and test")

# c) Every type in every part, and the smallest one
smallest_count = None
for combo in combos:
    in_train = (labels[train_idx] == combo).all(axis=1).sum()
    in_val = (labels[val_idx] == combo).all(axis=1).sum()
    in_test = (labels[test_idx] == combo).all(axis=1).sum()
    assert in_train > 0 and in_val > 0 and in_test > 0, "a type is missing from a part: " + wafer.label_name(combo)

    if smallest_count is None or in_train + in_val + in_test < smallest_count:
        smallest_count = in_train + in_val + in_test
        smallest = (wafer.label_name(combo), in_train, in_val, in_test)

print("All", len(combos), "types are in every part")
print(f"Smallest type: {smallest[0]}, training {smallest[1]}, validation {smallest[2]}, test {smallest[3]}")

No wafer is in two parts


No map is in training and test
All 38 types are in every part
Smallest type: Near-Full, training 93, validation 20, test 21


## 6. Same seed, same split? (QC-06)

**Why this step:** It must give the same result twice. If the split changed every run I couldn't compare scores between runs.

**The problem:** Make the split several times and compare with the one above.

**How:** `wafer.make_split` is exactly the loop from step 4, put in `wafer.py` so notebook 08 can run it again too. `np.array_equal(a, b)` is True when two arrays are exactly the same.

In [6]:
first = wafer.make_split(labels, SEED)
second = wafer.make_split(labels, SEED)

same_as_loop = (np.array_equal(first[0], train_idx) and np.array_equal(first[1], val_idx)
                and np.array_equal(first[2], test_idx))
same_twice = (np.array_equal(first[0], second[0]) and np.array_equal(first[1], second[1])
              and np.array_equal(first[2], second[2]))

print("Same as the loop above:", same_as_loop)
print("Same twice:", same_twice)
assert same_as_loop and same_twice, "QC-06 fails: the same seed gave a different split"

Same as the loop above: True
Same twice: True


## 7. Save it

**Why this step:** Future notebooks all read this file. None of them can clean or split slightly differently.

**The problem:** Save the cleaned maps, their labels and the three lists of positions in one file.

**How:** `np.savez_compressed` saves several arrays in one file, packed small. Wafer maps are mostly the same values over and over, so they pack really well. `os.path.getsize` gives the file size in bytes.

In [7]:
os.makedirs("output", exist_ok=True)
np.savez_compressed(
    "output/split.npz",
    maps=maps,
    labels=labels,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    seed=SEED,
)

print(f"Saved output/split.npz ({os.path.getsize('output/split.npz') / 1e6:.1f} MB)")

Saved output/split.npz (14.6 MB)


## Functions in this notebook

| Function | What it does | Where to read more |
|---|---|---|
| `.astype` | changes the number type, here to save memory | [docs](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html) |
| `np.unique(..., return_index=True)` | every different row once, and where it first appears | [docs](https://numpy.org/doc/stable/reference/generated/numpy.unique.html) |
| `np.sort` | puts numbers in order | [docs](https://numpy.org/doc/stable/reference/generated/numpy.sort.html) |
| `rng.shuffle` | mixes up a list | [docs](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.shuffle.html) |
| `np.intersect1d` | what two lists have in common | [docs](https://numpy.org/doc/stable/reference/generated/numpy.intersect1d.html) |
| `np.concatenate` | puts arrays after each other | [docs](https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html) |
| `np.array_equal` | True if two arrays are exactly the same | [docs](https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html) |
| `np.savez_compressed` | saves several arrays in one small file | [docs](https://numpy.org/doc/stable/reference/generated/numpy.savez_compressed.html) |
| `assert` | stops with a message when a check fails | [docs](https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement) |